In [1]:
import pandas as pd
import numpy as np
import os
from nltk.tokenize import word_tokenize

In [2]:
from IPython.display import HTML
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [3]:
#Setting working directory
#Desktop
os.chdir("C:/Users/john/Documents/GitHub_Data/VCAP-Open-Hours-Project-Data")
#Laptop
#os.chdir('C:/Users/johnc/Documents/GitHub_Data/VCAP-Open-Hours-Project-Data')

### READ ME

In [8]:
## Update, I am grabbing census tracts and counties and states for other clinics

In [11]:
other_addresses = pd.read_csv("other_geo_coded_addresses.csv")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8902 entries, 0 to 8901
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      8902 non-null   int64  
 1   full_address    8902 non-null   object 
 2   lat             8902 non-null   float64
 3   long            8902 non-null   float64
 4   google_address  8902 non-null   object 
 5   place_id        8902 non-null   object 
dtypes: float64(2), int64(1), object(3)
memory usage: 417.4+ KB


The purpose of this script is to add the census tract to the vet clinic address. Sometimes the api times out, so I need to be able to continually append to the same list

In [17]:
census_tract_join_dict = {"full_address" : [],
                           "full_census_tract" : []}

In [37]:
new_table = other_addresses.loc[8368:]
new_table.head()

,Unnamed: 0,full_address,lat,long,google_address,place_id
8368,8368,"SMALL ANIMAL HOSPITAL INC, 2340 N NEWHALL ST, ...",43.061010,-87.890108,"2340 N Newhall St, Milwaukee, WI 53211, USA",ChIJabs9ht4YBYgRqwHDgg-U5WE
8369,8369,"CENTRE EQUINE PRACTICE, 164 TUSSEY SINK RD, CE...",40.786034,-77.698906,"164 Tussey Sink Rd, Centre Hall, PA 16828, USA",ChIJb39Wv326zokR0-WtNQz0-BU
8370,8370,"LOYALSOCK ANIMAL HOSPITAL INC, 1900 NORTHWAY R...",41.261638,-76.969927,"1900 Northway Rd, Williamsport, PA 17701, USA",ChIJuYqa_9Olz4kRd71_7-iuepE
8371,8371,"CORY WILLIAMS DVM PLLC, 5866 ATHENS WALNUT HIL...",37.944133,-84.379072,"5866 Athens Walnut Hill Rd, Lexington, KY 4051...",ChIJyZ_vODlRQogRnTibUI1QkHA
8372,8372,"SONO VET VISION LLC, 2413 KAYLONNI LN, VAN BUR...",35.463203,-94.399951,"2413 Kaylonni Ln, Van Buren, AR 72956, USA",ChIJC5q4iMpKyocRDKVlKReoUzg


In [14]:
new_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8902 entries, 0 to 8901
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      8902 non-null   int64  
 1   full_address    8902 non-null   object 
 2   lat             8902 non-null   float64
 3   long            8902 non-null   float64
 4   google_address  8902 non-null   object 
 5   place_id        8902 non-null   object 
dtypes: float64(2), int64(1), object(3)
memory usage: 417.4+ KB


In [38]:
import censusgeocode as cg 

for ad, lat, lng in zip(new_table["full_address"], new_table["lat"], new_table["long"]):
    try:
        result = cg.coordinates(x=lng, y=lat)
        full_census_tract = result["Census Tracts"][0]["NAME"] + ", " + result['Counties'][0]['NAME'] + ", " + result["States"][0]['NAME']
        census_tract_join_dict["full_address"].append(ad)
        census_tract_join_dict["full_census_tract"].append(full_census_tract)
    except ValueError:
        census_tract_join_dict["full_address"].append(ad)
        census_tract_join_dict["full_census_tract"].append("Not Found")

In [40]:
other_address_tracts = pd.DataFrame(census_tract_join_dict)
other_address_tracts.tail()
len(other_address_tracts)

,full_address,full_census_tract
8897,"TIMBERLYNE ANIMAL CLINIC INC, 1703 LEGION RD #...","Census Tract 121.01, Orange County, North Caro..."
8898,"SOUTH TX VET SPECLSTS LLP, 503 E SONTERRA BLVD...","Census Tract 1918.18, Bexar County, Texas"
8899,"KEVIN A MONCE DVM, 305 ASHVILLE AVE, CARY, NC,...","Census Tract 535.09, Wake County, North Carolina"
8900,"PLANTATION ANIMAL HOSP-MTTHWS, 3036 WEDDINGTON...","Census Tract 58.51, Mecklenburg County, North ..."
8901,"ALT VET IMAGING, 7629 UNIVERSITY DR, SHREVEPOR...","Census Tract 239.04, Caddo Parish, Louisiana"


8902

In [41]:
other_addresses = other_addresses.merge(other_address_tracts, on='full_address', how='left')

In [45]:
other_addresses.to_csv("other_geo_coded_addresses.csv")

In [4]:
#Reading in Data

In [5]:
#All addresses list
all_addresses = pd.read_csv("all_geocoded_addresses.csv")
#census tract join table
#census_tract_join_table = pd.read_csv("census_tract_join_table.csv")

In [6]:
all_addresses_join_table = all_addresses[["full_address", "lat", "long"]]

In [7]:
len(all_addresses_join_table)

23637

In [32]:
#census_tract_join_dict = {"full_address" : [],
#                           "full_census_tract" : []}

In [47]:
len(census_tract_join_dict['full_address'])

16115

In [54]:
new_table = all_addresses_join_table.loc[16115:]

In [55]:
new_table.head()

,full_address,lat,long
16115,"ALVIN-FRIENDSWOOD VET CLINIC, 2407 W PARKWOOD ...",29.482395,-95.217793
16116,"EASTON COMMONS ANIMAL HOSPITAL, 15030 WEST RD,...",29.900835,-95.628488
16117,"VCA SOUTHWEST FREEWAY ANIMAL, 15575 SOUTHWEST ...",29.602112,-95.617226
16118,"VALLEY ANIMAL HOSPITAL LLC, 520 E RIDGE RD, PA...",40.320400,-76.595854
16119,"PETVET, 6763 HIGHWAY 6 S # 1800, HOUSTON, TX, ...",29.704405,-95.642775


In [56]:
import censusgeocode as cg 

for ad, lat, lng in zip(new_table["full_address"], new_table["lat"], new_table["long"]):
    try:
        result = cg.coordinates(x=lng, y=lat)
        full_census_tract = result["Census Tracts"][0]["NAME"] + ", " + result['Counties'][0]['NAME'] + ", " + result["States"][0]['NAME']
        census_tract_join_dict["full_address"].append(ad)
        census_tract_join_dict["full_census_tract"].append(full_census_tract)
    except ValueError:
        census_tract_join_dict["full_address"].append(ad)
        census_tract_join_dict["full_census_tract"].append("Not Found")

,full_address,full_census_tract
23632,"HEALING HANDS VETERINARY HOSP, 7427 N LOOP 160...","Census Tract 1820.03, Bexar County, Texas"
23633,"BROOKWOOD ANIMAL HOSP-JENNIFER, 922 DOGWOOD RD...","Census Tract 507.56, Gwinnett County, Georgia"
23634,"SHORE KAREN BVMS, 5280 PACHECO BLVD, PACHECO, ...","Census Tract 3212, Contra Costa County, Califo..."
23635,"BORGFELD ANIMAL HOSPITAL PLLC, 1304 W BORGFELD...","Census Tract 1918.04, Bexar County, Texas"
23636,"ATLANTIC ANIMAL HOSPITAL PA, 1319 MILITARY CUT...","Census Tract 106, New Hanover County, North Ca..."


In [62]:
census_tract_join_table = pd.DataFrame(census_tract_join_dict)

In [63]:
census_tract_join_table.to_csv("census_tract_join_table.csv")

In [68]:
import censusgeocode as cg 

cg.coordinates(x = -73.25840, y = 41.31197)

{'States': [{'STATENS': '01779780',
   'GEOID': '09',
   'CENTLAT': '+41.5751437',
   'AREAWATER': 1816364426,
   'STATE': '09',
   'BASENAME': 'Connecticut',
   'STUSAB': 'CT',
   'OID': '27490331955805',
   'LSADC': '00',
   'FUNCSTAT': 'A',
   'INTPTLAT': '+41.5798637',
   'DIVISION': '1',
   'NAME': 'Connecticut',
   'REGION': '1',
   'OBJECTID': 26,
   'CENTLON': '-072.7393081',
   'AREALAND': 12541750274,
   'INTPTLON': '-072.7466572',
   'MTFCC': 'G4000',
   'CENT': (-72.7393081, 41.5751437),
   'INTPT': (-72.7466572, 41.5798637)}],
 'County Subdivisions': [{'COUSUB': '48620',
   'GEOID': '0912048620',
   'CENTLAT': '+41.3379086',
   'AREAWATER': 511097,
   'STATE': '09',
   'BASENAME': 'Monroe',
   'OID': '27690697821317',
   'LSADC': '43',
   'FUNCSTAT': 'A',
   'INTPTLAT': '+41.3392358',
   'NAME': 'Monroe town',
   'OBJECTID': 23356,
   'CENTLON': '-073.2250242',
   'COUSUBCC': 'T1',
   'AREALAND': 67522444,
   'INTPTLON': '-073.2228275',
   'MTFCC': 'G4040',
   'COUSUBNS': 

In [64]:
from censusgeocode import CensusGeocode
cg = CensusGeocode(benchmark='Public_AR_Current', vintage='Census2020_Current')

In [65]:
cg.coordinates(x = -95.53871, y = 30.17833)

{'2018 State Legislative Districts - Upper': [{'POP100': 1019150,
   'GEOID': '48004',
   'CENTLAT': '+29.8772005',
   'AREAWATER': 1972985466,
   'STATE': '48',
   'BASENAME': '4',
   'OID': '212704690779008',
   'LSADC': 'LU',
   'SLDU': '004',
   'FUNCSTAT': 'N',
   'INTPTLAT': '+29.8749870',
   'NAME': 'State Senate District 4',
   'OBJECTID': 771,
   'CENTLON': '-094.7002502',
   'LSY': '2018',
   'AREALAND': 6313883172,
   'INTPTLON': '-094.7127170',
   'HU100': 396829,
   'MTFCC': 'G5210',
   'LDTYP': 'O',
   'CENT': (-94.7002502, 29.8772005),
   'INTPT': (-94.712717, 29.874987)}],
 'States': [{'STATENS': '01779801',
   'POP100': 29145505,
   'GEOID': '48',
   'CENTLAT': '+31.4277553',
   'AREAWATER': 18974670781,
   'STATE': '48',
   'BASENAME': 'Texas',
   'STUSAB': 'TX',
   'OID': '2747098452115',
   'LSADC': '00',
   'FUNCSTAT': 'A',
   'INTPTLAT': '+31.4347032',
   'DIVISION': '7',
   'NAME': 'Texas',
   'REGION': '3',
   'OBJECTID': 35,
   'CENTLON': '-099.2890828',
   'AR